In [ ]:
# Src/vcd_decoding.ipynb 에 들어갈 내용
import requests

OLLAMA_URL = 'http://localhost:11434/api/generate'

def generate_llava_response(prompt, image_b64, temperature=0.0):
    payload = {
        "model": "llava",
        "prompt": prompt,
        "images": [image_b64],
        "stream": False,
        "options": {"temperature": temperature}
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload)
        return response.json().get('response', 'Error')
    except Exception as e:
        return f"Error: {e}"

# 이 함수가 정의되어 있어야 03번 파일에서 에러가 안 납니다.
def generate_vcd_candidates(question, orig_b64, noisy_b64):
    prompt = f"You are a helpful assistant for a visually impaired person. Please answer the question based on the image.\nQuestion: {question}"
    
    # 1. 원본 (Baseline 재활용이 아닌 새로 뽑는 버전일 때)
    c1 = generate_llava_response(prompt, orig_b64, temperature=0.0)
    # 2. 도메인 왜곡 (B-VCD)
    c2 = generate_llava_response(prompt, noisy_b64, temperature=0.0)
    # 3. 창의적 답변
    c3 = generate_llava_response(prompt, orig_b64, temperature=0.7)
    
    return {
        "candidate_1_regular": c1,
        "candidate_2_noisy": c2,
        "candidate_3_diverse": c3
    }